In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import subprocess
import os
from dataclasses import dataclass

@dataclass
class RunConfig:
    alg_name: str
    array_size: int         = 2**20         # n
    k_value: int | None     = 2             # k
    threshold: int          = 2**10         # threshold
    num_threads: int | None = 1             # n_threads
    epochs: int             = 5             # repeticiones


In [2]:
def run_single_experiment(config: RunConfig):
    """
    Cada algoritmo recibe un input (n, k, threshold).
    En caso de que un parametro no corresponda, se ignora.
    El numero de hebras se saca de env
    """
    
    env_dict = os.environ.copy()
    env_dict["OMP_NUM_THREADS"] = str(config.num_threads)

    alg_bin_path = "./bin/" + config.alg_name
    array_size = str(config.array_size)
    k_value = str(config.k_value)
    threshold = str(config.threshold)
    args = [alg_bin_path, array_size, k_value, threshold]

    # Se lanza el comando con los argumentos anteriores, captura el output como texto
    # Le pasa el env modificado anteriormente y revisa el retorno por si hay error
    output = subprocess.run(args=args, capture_output=True, text=True, env=env_dict, check=True)

    return output




In [3]:
def process_experiment(exp: str):
    """
    formato: nombre_algoritmo,n,k,threshold,threads,is_sorted,tiempo_segundos.
    """
    split_exp = exp.split(',')
    elapsed_time = float(split_exp[-1])
    return elapsed_time



In [4]:
def run_with_epocs(config: RunConfig):

    times = []
    for i in range(config.epochs):
        output = run_single_experiment(config).stdout.strip()
        times.append(process_experiment(output))

    mean_time = np.mean(times)
    std_time = np.std(times)

    output_dict = {
        "alg_name": config.alg_name,
        "array_size": config.array_size,
        "k_value": config.k_value,
        "threshold": config.threshold,
        "num_threads": config.num_threads,
        "mean_time": mean_time,
        "std_time": std_time
    }

    return output_dict
    

In [ ]:
THREAD_VALUES = [1, 2, 4, 8]
N_VALUES = [2**20, 2**22, 2**24, 2**26]
K_VALUES = [2**1, 2**2, 2**3, 2**4, 2**5]
THRESHOLD_VALUES = [2**4, 2**6, 2**8, 2**10, 2**12]
EPOCHS = 5

lista_resultados = []

In [6]:
"""
Secuencial Clasico: mergesort_seq
formato: nombre_algoritmo,n,-1,threshold,1,is_sorted,tiempo_segundos.
"""

for n in N_VALUES:
    for threshold in THRESHOLD_VALUES:
        cfg = RunConfig("mergesort_seq", n, -1, threshold, 1, EPOCHS)
        resultado = run_with_epocs(cfg)
        lista_resultados.append(resultado)
        print(f"Resultado para n={n}, threshold={threshold}: {resultado}")



Resultado para n=1048576, threshold=16: {'alg_name': 'mergesort_seq', 'array_size': 1048576, 'k_value': -1, 'threshold': 16, 'num_threads': 1, 'mean_time': np.float64(0.07755614), 'std_time': np.float64(0.0016074904374210148)}
Resultado para n=1048576, threshold=64: {'alg_name': 'mergesort_seq', 'array_size': 1048576, 'k_value': -1, 'threshold': 64, 'num_threads': 1, 'mean_time': np.float64(0.07860323999999999), 'std_time': np.float64(0.0014900724057575227)}
Resultado para n=1048576, threshold=256: {'alg_name': 'mergesort_seq', 'array_size': 1048576, 'k_value': -1, 'threshold': 256, 'num_threads': 1, 'mean_time': np.float64(0.07621697999999999), 'std_time': np.float64(0.0006792891502151339)}
Resultado para n=1048576, threshold=1024: {'alg_name': 'mergesort_seq', 'array_size': 1048576, 'k_value': -1, 'threshold': 1024, 'num_threads': 1, 'mean_time': np.float64(0.07449874000000001), 'std_time': np.float64(0.0009222257979475525)}
Resultado para n=1048576, threshold=4096: {'alg_name': 'mer

In [7]:
"""
Paralelo Clasico: mergesort_par
formato: nombre_algoritmo,n,-1,threshold,num_threads,is_sorted,tiempo_segundos.
"""

for n in N_VALUES:
    for threshold in THRESHOLD_VALUES:
        for num_threads in THREAD_VALUES:
            cfg = RunConfig("mergesort_par", n, -1, threshold, num_threads, EPOCHS)
            resultado = run_with_epocs(cfg)
            lista_resultados.append(resultado)
            print(f"Resultado para n={n}, threshold={threshold}, num_threads={num_threads}: {resultado}")



Resultado para n=1048576, threshold=16, num_threads=1: {'alg_name': 'mergesort_par', 'array_size': 1048576, 'k_value': -1, 'threshold': 16, 'num_threads': 1, 'mean_time': np.float64(0.08847568), 'std_time': np.float64(0.0018803918032154908)}
Resultado para n=1048576, threshold=16, num_threads=2: {'alg_name': 'mergesort_par', 'array_size': 1048576, 'k_value': -1, 'threshold': 16, 'num_threads': 2, 'mean_time': np.float64(0.0673936), 'std_time': np.float64(0.003085007052179945)}
Resultado para n=1048576, threshold=16, num_threads=4: {'alg_name': 'mergesort_par', 'array_size': 1048576, 'k_value': -1, 'threshold': 16, 'num_threads': 4, 'mean_time': np.float64(0.06971434000000001), 'std_time': np.float64(0.0032950644076254415)}
Resultado para n=1048576, threshold=16, num_threads=8: {'alg_name': 'mergesort_par', 'array_size': 1048576, 'k_value': -1, 'threshold': 16, 'num_threads': 8, 'mean_time': np.float64(0.07378898), 'std_time': np.float64(0.0033875489649007277)}
Resultado para n=1048576,

In [9]:
"""
Secuencial K-way: k-way_seq
formato: nombre_algoritmo,n,k,threshold,1,is_sorted,tiempo_segundos.
"""
for n in N_VALUES:
    for k in K_VALUES:
        for threshold in THRESHOLD_VALUES:
            cfg = RunConfig("kway_seq", n, k, threshold, 1, EPOCHS)
            resultado = run_with_epocs(cfg)
            lista_resultados.append(resultado)
            print(f"Resultado para n={n}, k={k}, threshold={threshold}: {resultado}")



Resultado para n=1048576, k=2, threshold=16: {'alg_name': 'kway_seq', 'array_size': 1048576, 'k_value': 2, 'threshold': 16, 'num_threads': 1, 'mean_time': np.float64(0.270428), 'std_time': np.float64(0.0023185200452012426)}
Resultado para n=1048576, k=2, threshold=64: {'alg_name': 'kway_seq', 'array_size': 1048576, 'k_value': 2, 'threshold': 64, 'num_threads': 1, 'mean_time': np.float64(0.24146300000000004), 'std_time': np.float64(0.001927216957169068)}
Resultado para n=1048576, k=2, threshold=256: {'alg_name': 'kway_seq', 'array_size': 1048576, 'k_value': 2, 'threshold': 256, 'num_threads': 1, 'mean_time': np.float64(0.21728579999999997), 'std_time': np.float64(0.0034885230915102164)}
Resultado para n=1048576, k=2, threshold=1024: {'alg_name': 'kway_seq', 'array_size': 1048576, 'k_value': 2, 'threshold': 1024, 'num_threads': 1, 'mean_time': np.float64(0.193615), 'std_time': np.float64(0.001131380572574935)}
Resultado para n=1048576, k=2, threshold=4096: {'alg_name': 'kway_seq', 'array

KeyboardInterrupt: 

In [11]:
THREAD_VALUES = [2, 4, 8]

In [12]:
"""
Paralelos K-way: k-way_par, k_way_ranks, k-way_full
formato: nombre_algoritmo,n,k,threshold,num_threads,is_sorted,tiempo_seg
"""

for alg_name in ("kway_par", "kway_ranks", "kway_full"):
    for n in N_VALUES:
        for k in K_VALUES:
            for threshold in THRESHOLD_VALUES:
                for num_threads in THREAD_VALUES:
                    cfg = RunConfig(alg_name, n, k, threshold, num_threads, EPOCHS)
                    resultado = run_with_epocs(cfg)
                    lista_resultados.append(resultado)
                    print(f"Resultado para alg={alg_name}, n={n}, k={k}, threshold={threshold}, num_threads={num_threads}: {resultado}")

Resultado para alg=kway_par, n=1048576, k=2, threshold=16, num_threads=2: {'alg_name': 'kway_par', 'array_size': 1048576, 'k_value': 2, 'threshold': 16, 'num_threads': 2, 'mean_time': np.float64(0.1757466), 'std_time': np.float64(0.003666927847667583)}
Resultado para alg=kway_par, n=1048576, k=2, threshold=16, num_threads=4: {'alg_name': 'kway_par', 'array_size': 1048576, 'k_value': 2, 'threshold': 16, 'num_threads': 4, 'mean_time': np.float64(0.1538678), 'std_time': np.float64(0.006362245905338777)}
Resultado para alg=kway_par, n=1048576, k=2, threshold=16, num_threads=8: {'alg_name': 'kway_par', 'array_size': 1048576, 'k_value': 2, 'threshold': 16, 'num_threads': 8, 'mean_time': np.float64(0.11923500000000001), 'std_time': np.float64(0.0031189862455612053)}
Resultado para alg=kway_par, n=1048576, k=2, threshold=64, num_threads=2: {'alg_name': 'kway_par', 'array_size': 1048576, 'k_value': 2, 'threshold': 64, 'num_threads': 2, 'mean_time': np.float64(0.1460498), 'std_time': np.float64(

In [13]:
df = pd.DataFrame(lista_resultados)
df.to_csv("resultados.csv", index=False)